In [1]:
# scripts/03_dedupe.ipynb

import sys
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

# 1. Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading MPNet on {device}...")
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

# 2. The "Torture Test" Dataset
# Expanded to include high-risk edge cases for News/Financial data.
sentences = [
    # --- Group A: The Easy Stuff (Lexical) ---
    "The inflation rate is 3.5%.",                        # Anchor 1
    "The inflation rate is 3.5%.",                        # Exact
    "The inflation rate is 3.50%.",                       # Lexical (Decimal)
    
    # --- Group B: True Paraphrases (Target: DELETE) ---
    "The central bank hiked interest rates.",             # Anchor 2
    "Interest rates were raised by the central bank.",    # Passive Voice
    "The Federal Reserve increased borrowing costs.",     # Semantic Substitution (Hard)
    
    # --- Group C: The "Death Zone" (High Similarity, Different Meaning) ---
    # !! CRITICAL: If we delete these, we lose facts.
    "The deal has been signed.",                          # Anchor 3
    "The deal has not been signed.",                      # Negation (Vectors often confuse this!)
    
    "Revenue grew by 5% last quarter.",                   # Anchor 4
    "Revenue grew by 50% last quarter.",                  # Numerical Deviation
    
    "Google acquired Fitbit.",                            # Anchor 5
    "Fitbit acquired Google.",                            # Subject/Object Swap
    
    # --- Group D: Contextual Drift ---
    "The movie was excellent.",
    "The film was a masterpiece.",                        # Soft Paraphrase
    "The movie was terrible."                             # Antonym (Should be kept?)
]

print(f"Test Dataset: {len(sentences)} sentences")

# ==========================================
# METHOD 1: Lexical Similarity (Jaccard)
# ==========================================
def get_jaccard_sim(str1, str2):
    a = set(str1.lower().split())
    b = set(str2.lower().split())
    c = a.intersection(b)
    return float(len(c)) / (len(a) + len(b) - len(c))

print("\n--- Method 1: Lexical (Jaccard > 0.5) ---")
kept_lexical = []
for s in sentences:
    is_dupe = False
    for k in kept_lexical:
        score = get_jaccard_sim(s, k)
        if score > 0.5: 
            is_dupe = True
            # Check if it failed on Negation
            if "not" in s and "not" not in k:
                print(f"  [DANGER] Dropped Negation: '{s}' (Sim {score:.2f} with '{k}')")
            else:
                print(f"  [DROP] '{s}' (Sim {score:.2f} with '{k}')")
            break
    if not is_dupe:
        kept_lexical.append(s)

# ==========================================
# METHOD 2: Brute-Force Vector (Cosine Matrix)
# ==========================================
print("\n--- Method 2: Brute-Force Vector (Cosine > 0.90) ---")
# Note: We raised threshold to 0.90 to try and save the "Negation" cases.

embeddings = model.encode(sentences, convert_to_tensor=True)
cosine_scores = util.cos_sim(embeddings, embeddings)

kept_vector = []
indices_to_remove = set()

for i in range(len(sentences)):
    if i in indices_to_remove:
        continue
    
    kept_vector.append(sentences[i])
    
    for j in range(i + 1, len(sentences)):
        if j in indices_to_remove:
            continue
            
        score = cosine_scores[i][j].item()
        
        # LOGIC: High Similarity
        if score > 0.90:
            print(f"  [DROP] '{sentences[j]}' (Score {score:.4f} with '{sentences[i]}')")
            indices_to_remove.add(j)
            
        # WARNING LOGIC: Check "Near Misses" in the Dangerous Zone (0.80 - 0.90)
        elif score > 0.80:
             print(f"  [WARNING] High Sim ({score:.4f}): '{sentences[i]}' vs '{sentences[j]}'")

# ==========================================
# METHOD 3: SemHash / Fast Clustering
# ==========================================
print("\n--- Method 3: Fast Clustering (Threshold 0.90) ---")
# min_community_size=1 means we get every sentence back, organized into clusters
clusters = util.community_detection(embeddings, min_community_size=1, threshold=0.90)

kept_clustering = []
for i, cluster in enumerate(clusters):
    rep_idx = cluster[0]
    kept_clustering.append(sentences[rep_idx])
    
    if len(cluster) > 1:
        duplicates = [sentences[idx] for idx in cluster[1:]]
        print(f"  Cluster {i+1}: Kept '{sentences[rep_idx]}'. Dropped: {duplicates}")

# ==========================================
# FINAL REPORT
# ==========================================
print("\n" + "="*40)
print(f"Original: {len(sentences)}")
print(f"Lexical:  {len(kept_lexical)}")
print(f"Vector:   {len(kept_vector)}")
print(f"Cluster:  {len(kept_clustering)}")

# Negation Safety Check
negation_survived = any("not been signed" in s for s in kept_vector)
print(f"\nSAFETY CHECK: Did 'The deal has NOT been signed' survive Vector cleanup? {negation_survived}")
if not negation_survived:
    print("CRITICAL FAIL: The model deleted the negation. You MUST raise the threshold > 0.95.")

/opt/conda/envs/sentinel-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading MPNet on cpu...
Test Dataset: 15 sentences

--- Method 1: Lexical (Jaccard > 0.5) ---
  [DROP] 'The inflation rate is 3.5%.' (Sim 1.00 with 'The inflation rate is 3.5%.')
  [DROP] 'The inflation rate is 3.50%.' (Sim 0.67 with 'The inflation rate is 3.5%.')
  [DANGER] Dropped Negation: 'The deal has not been signed.' (Sim 0.83 with 'The deal has been signed.')
  [DROP] 'Revenue grew by 50% last quarter.' (Sim 0.71 with 'Revenue grew by 5% last quarter.')
  [DROP] 'The movie was terrible.' (Sim 0.60 with 'The movie was excellent.')

--- Method 2: Brute-Force Vector (Cosine > 0.90) ---
  [DROP] 'The inflation rate is 3.5%.' (Score 1.0000 with 'The inflation rate is 3.5%.')
  [DROP] 'The inflation rate is 3.50%.' (Score 0.9833 with 'The inflation rate is 3.5%.')
  [DROP] 'Interest rates were raised by the central bank.' (Score 0.9226 with 'The central bank hiked interest rates.')
  [DROP] 'Fitbit acquired Google.' (Score 0.9768 with 'Google acquired Fitbit.')

--- Method 3: Fast Cl

In [3]:
# scripts/03_dedupe.ipynb

import sys
import torch
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer, util

# 1. Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading MPNet on {device}...")
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

# 2. Hybrid Logic Definition
def is_lexically_safe(s1, s2):
    """
    Returns True if it is 'safe' to merge these two sentences.
    Checks for:
    1. Negation mismatches (not, never, no)
    2. Numerical mismatches (3% vs 30%)
    """
    text1 = s1.lower()
    text2 = s2.lower()
    
    # A. Negation Check
    negations = {"not", "no", "never", "n't"}
    s1_has_neg = any(n in text1.split() for n in negations)
    s2_has_neg = any(n in text2.split() for n in negations)
    
    if s1_has_neg != s2_has_neg:
        return False, "Negation Mismatch"
        
    # B. Numbers Check
    # Extract numbers (integers and floats)
    nums1 = set(re.findall(r'\d+\.?\d*', text1))
    nums2 = set(re.findall(r'\d+\.?\d*', text2))
    
    # If both have numbers, they must match perfectly
    if nums1 and nums2 and nums1 != nums2:
        return False, f"Number Mismatch ({nums1} vs {nums2})"
    
    return True, "Safe"

# 3. Test Data (The Hard Cases)
sentences = [
    "The deal has been signed.",
    "The deal has not been signed.",         # Negation Fail
    "Inflation is at 3.0%.",
    "Inflation is at 30%.",                  # Number Fail
    "The economy is growing.",
    "The economy is expanding.",             # Paraphrase (Should Merge)
    "Google bought Fitbit.",
    "Fitbit bought Google."                  # Subject Swap (Hard to catch, but vectors usually separate them slightly)
]

print(f"Test Dataset: {len(sentences)} sentences")

# 4. Run Hybrid Pipeline
print("\n--- Running Hybrid Pipeline ---")

# Step A: Semantic Clustering
embeddings = model.encode(sentences, convert_to_tensor=True)
# High threshold for initial grouping
clusters = util.community_detection(embeddings, min_community_size=1, threshold=0.85) 

kept_sentences = []

for cluster in clusters:
    # cluster is list of indices [0, 1, ...]
    anchor_idx = cluster[0]
    anchor_text = sentences[anchor_idx]
    
    kept_sentences.append(anchor_text)
    
    # Check candidates for merging
    if len(cluster) > 1:
        for duplicate_idx in cluster[1:]:
            duplicate_text = sentences[duplicate_idx]
            
            # Step B: Hybrid Safety Gate
            safe, reason = is_lexically_safe(anchor_text, duplicate_text)
            
            if safe:
                print(f"  [MERGE] '{duplicate_text}' -> '{anchor_text}'")
            else:
                print(f"  [KEEP] '{duplicate_text}' (Vector sim high, but {reason})")
                kept_sentences.append(duplicate_text)

print("\n" + "="*40)
print(f"Final List ({len(kept_sentences)}):")
for s in kept_sentences:
    print(f" - {s}")

Loading MPNet on cpu...
Test Dataset: 8 sentences

--- Running Hybrid Pipeline ---
  [MERGE] 'The economy is expanding.' -> 'The economy is growing.'
  [MERGE] 'Fitbit bought Google.' -> 'Google bought Fitbit.'

Final List (6):
 - The economy is growing.
 - Google bought Fitbit.
 - The deal has been signed.
 - The deal has not been signed.
 - Inflation is at 3.0%.
 - Inflation is at 30%.
